In [49]:
using LowLevelFEM, LinearAlgebra

In [50]:
openGeometry("boxes.geo")

In [51]:
#openPreProcessor()

In [52]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u);

In [53]:
bc_bottom = BoundaryCondition("bottom", ux=0, uy=0, uz=0)
bc_top = BoundaryCondition("top", ux=0, uz=0, uy=(x,y,z)->-x*(x-10) * z*(z-10) / 5250)

K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U))
f = ∫(U ⋅ [0, 0, 0])

@time u = solveField(Symmetric(K), f, support=[bc_bottom, bc_top])

showDoFResults(u, name="u", factor=1, visible=false)

  0.423498 seconds (115.75 k allocations: 50.401 MiB, 2.60% gc time, 24.26% compilation time)


0

In [54]:
contact_pair = contact(u, master="master", slave="slave", cn=1e8, topology_tol=0.0)

Contact("slave" -> "master", 789 candidate nodes, 28 active, G=(2367, 9429), C=(2367, 2367))

In [55]:
using SparseArrays

# Lagrange multiplier field
Λ = Field([mat], type=:VectorField, dim=3, fieldName=:λ)

# Contact kinematics
L = contact(
    u,
    master="master",
    slave="slave",
    LagrangeMultiplierField=Λ,
    topology_tol=0.05,
    projection_tol=0.001
)

support = [bc_bottom, bc_top]
free_u = freeDoFs(U, support)

u_it = copy(u)
λ_it = vectorField(Λ, "body", [0, 0, 0])

# Zero multiplier block
nλ = size(L.E, 1)
Zλ = SystemMatrix(spzeros(nλ, nλ), Λ)

# Primal-dual active-set parameter.
# This is NOT a penalty stiffness; it is used only to determine the active set.
κ = 1e8

active_old = falses(length(L.slave_nodes))
active_old2 = falses(length(L.slave_nodes))

for iter in 1:40

    # Current contact geometry
    updateContact!(L, u_it)

    (; G, g, E) = L

    pdim = L.U.pdim

    # Normal component in the reduced contact space
    normal_rows = 1:pdim:length(g)

    # Corresponding normal multiplier DoFs
    λn_dofs = L.multiplier_dofs[normal_rows]
    λn = DoFs(λ_it)[λn_dofs]

    # ----------------------------------------------------------
    # Primal-dual active set
    #
    # Sign convention:
    #     g_n >= 0       open/admissible
    #     λ_n <= 0       compression
    #
    # active <=> λ_n + κ g_n < 0
    # ----------------------------------------------------------
    active = λn .+ κ .* L.gap_values .< 0.0

    # Detect A -> B -> A cycling
    two_cycle =
        iter > 2 &&
        active == active_old2 &&
        active != active_old

    if two_cycle
        switching = active .!= active_old

        println(
            "Two-cycle detected: freezing ",
            count(switching),
            " switching contact points."
        )

        active[switching] .= active_old[switching]
    end

    # Inactive multipliers are zero
    DoFs(λ_it)[λn_dofs[.!active]] .= 0.0

    # ----------------------------------------------------------
    # Active normal projector in contact space
    # ----------------------------------------------------------
    χ = zeros(Float64, length(g))
    χ[normal_rows[active]] .= 1.0

    P = SystemMatrix(
        spdiagm(0 => χ),
        nothing,
        nothing,
        nothing,
        nothing
    )

    # ----------------------------------------------------------
    # Contact constraint operator
    #
    #     B = E P G
    #     gλ = E P g
    # ----------------------------------------------------------
    B  = E * P * G
    gλ = E * P * g

    # ----------------------------------------------------------
    # KKT residual
    #
    #     rᵤ = K u - f + B' λ
    #     rλ = gλ
    # ----------------------------------------------------------
    rᵤ = K * u_it - f + B' * λ_it
    rλ = gλ

    # ----------------------------------------------------------
    # KKT tangent
    #
    #         [ K   B' ]
    #     A = [        ]
    #         [ B    0 ]
    # ----------------------------------------------------------
    A = SystemMatrix([
        K   B'
        B   Zλ
    ])

    r = SystemVector([rᵤ, rλ])

    # Offset of the multiplier field in the multifield system
    λoff = A.offsets[2]

    # Only displacement free DoFs and ACTIVE NORMAL multiplier DoFs
    # participate in the Newton correction.
    free = vcat(
        free_u,
        λoff .+ λn_dofs[active]
    )

    Δx = zeros(Float64, size(A, 1))

    Δx[free] =
        -A[free, free] \ r.a[free, 1]

    # Split the multifield correction
    Δu = @view Δx[1:λoff]
    Δλ = @view Δx[λoff+1:end]

    # Newton update
    DoFs(u_it)[:] .+= Δu
    DoFs(λ_it)[:] .+= Δλ

    # ----------------------------------------------------------
    # Convergence diagnostics
    # ----------------------------------------------------------
    Δactive = count(active .!= active_old)

    err_u =
        norm(Δu[free_u]) /
        max(norm(DoFs(u_it)[free_u]), eps())

    max_gap =
        any(active) ?
        maximum(abs.(L.gap_values[active])) :
        0.0

    λn = DoFs(λ_it)[λn_dofs]

    min_λ =
        any(active) ?
        minimum(λn[active]) :
        0.0

    println(
        "iter = ", iter,
        ", active = ", count(active),
        ", Δactive = ", Δactive,
        ", max |gap| = ", max_gap,
        ", min λn = ", min_λ,
        ", error = ", err_u
    )

    converged =
        Δactive == 0 &&
        max_gap < 1e-8 &&
        err_u < 1e-8

    active_old2 .= active_old
    active_old .= active

    converged && break
end

u_LM = u_it
λ_LM = λ_it

iter = 1, active = 28, Δactive = 28, max |gap| = 0.005301136455907254, min λn = -119.801502922777, error = 0.006108040885967168
iter = 2, active = 28, Δactive = 0, max |gap| = 3.107626036192155e-5, min λn = -119.99101200400085, error = 4.647259473970331e-5
iter = 3, active = 19, Δactive = 9, max |gap| = 1.2615468257426324e-7, min λn = -112.37674890755265, error = 0.0007729432498484765
iter = 4, active = 8, Δactive = 11, max |gap| = 7.805509443415984e-7, min λn = -141.0729781305081, error = 0.0019397781635831196
iter = 5, active = 23, Δactive = 15, max |gap| = 0.002264915883948991, min λn = -112.76747154032783, error = 0.0018416672761591558
iter = 6, active = 15, Δactive = 8, max |gap| = 1.638791145348833e-5, min λn = -162.39246768470875, error = 0.0018193515257132504
iter = 7, active = 10, Δactive = 21, max |gap| = 0.0026875945285431014, min λn = -150.35436622612104, error = 0.0023386393218810773
iter = 8, active = 22, Δactive = 18, max |gap| = 0.001965666832689414, min λn = -112.34075

nodal VectorField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [56]:
showDoFResults(u_LM, name="u cont.", visible=true, factor=1)


1

In [57]:
λ_LM = nodesToElements(λ_LM, onPhysicalGroup="slave")
showElementResults(λ_LM[1], name="λ")

2

In [58]:
λ_LM.type

:v3D

In [59]:
showElementResults(contact_pair.gap, name="gap")

3

In [60]:
openPostProcessor()

Két vagy több párnál majd:

```Julia
contacts = ContactSet(c1, c2, c3)

updateContact!(contacts, u_it)

Kc = sum(c.G' * c.C * c.G for c in contacts)
rc = sum(c.G' * c.C * c.g for c in contacts)

r = K * u_it - f + rc
A = K + Kc
```